In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
from src.data import load_region_df
from src.soc_engine import BatterySpec
from src.forecaster import (
    BATTERY_LZ_COL,
    SETTLEMENT_COL,
    train_houston_price_model,
    generate_day_ahead_forecast,
    generate_forecast_range,
    forecast_accuracy,
)
from src.dam_bidder import (
    DAMBid,
    optimise_dam_bid,
    settle_dam_bid,
    run_dam_campaign,
    dam_campaign_kpis,
)

passed = 0
failed = 0

def check(condition, msg_pass, msg_fail):
    global passed, failed
    if condition:
        print(f"✅ {msg_pass}")
        passed += 1
    else:
        print(f"❌ {msg_fail}")
        failed += 1

df_coast   = load_region_df("coast")
spec       = BatterySpec(power_mw=100, duration_hours=4)
test_start = pd.Timestamp("2025-01-01")

# ── Test 1 — Settlement column is always LZ_HOUSTON_DAM ───────────────────
check(
    BATTERY_LZ_COL  == "LZ_HOUSTON_DAM",
    "Test 1 — BATTERY_LZ_COL = LZ_HOUSTON_DAM",
    f"Test 1 — wrong: {BATTERY_LZ_COL}"
)
check(
    SETTLEMENT_COL == "LZ_HOUSTON_DAM",
    "Test 1 — SETTLEMENT_COL = LZ_HOUSTON_DAM",
    f"Test 1 — wrong: {SETTLEMENT_COL}"
)

# ── Test 2 — LZ_HOUSTON_DAM exists in df_coast ────────────────────────────
check(
    "LZ_HOUSTON_DAM" in df_coast.columns,
    "Test 2 — LZ_HOUSTON_DAM confirmed in df_coast",
    "Test 2 — LZ_HOUSTON_DAM MISSING from df_coast"
)

# ── Test 3 — train_houston_price_model targets LZ price not spread ─────────
print(f"\nTraining Houston price model (train < {test_start.date()})...")
model_result = train_houston_price_model(df_coast, test_start)

check(isinstance(model_result, dict),
      "Test 3 — model result is dict",
      "Test 3 — wrong type")
check("predictions_df" in model_result,
      "Test 3 — predictions_df present",
      "Test 3 — predictions_df MISSING")
check(
    "actual_houston_price" in model_result["predictions_df"].columns,
    "Test 3 — actual_houston_price column in predictions_df",
    "Test 3 — MISSING actual_houston_price"
)
check(
    "forecast_houston_price" in model_result["predictions_df"].columns,
    "Test 3 — forecast_houston_price column in predictions_df",
    "Test 3 — MISSING forecast_houston_price"
)
actuals = model_result["predictions_df"]["actual_houston_price"]
check(
    actuals.mean() > 0,
    f"Test 3 — target is LZ price (mean=${actuals.mean():.2f}) not spread",
    f"Test 3 — target looks like a spread (mean=${actuals.mean():.2f})"
)
print(f"  MAE: ${model_result['mae']:.2f} | RMSE: ${model_result['rmse']:.2f}")
print(f"  Train: {model_result['train_size']:,} hrs | Test: {model_result['test_size']:,} hrs\n")

In [ ]:
# ── Test 4 — No lookahead — test set only contains dates >= test_start ──────
pred_df = model_result["predictions_df"]
check(
    pred_df.index.min() >= test_start,
    f"Test 4 — test set starts at {pred_df.index.min().date()} >= {test_start.date()}",
    f"Test 4 — lookahead detected: test set starts {pred_df.index.min().date()}"
)

# ── Test 5 — generate_forecast_range returns correct structure ─────────────
start = pd.Timestamp("2025-06-01")
end   = pd.Timestamp("2025-06-07")
forecast_df = generate_forecast_range(model_result, df_coast, start, end)

check(
    "forecast_houston_price" in forecast_df.columns,
    "Test 5 — forecast_houston_price column present",
    "Test 5 — forecast_houston_price MISSING"
)
check(
    "actual_houston_price" in forecast_df.columns,
    "Test 5 — actual_houston_price column present",
    "Test 5 — actual_houston_price MISSING — needed for settlement comparison"
)
check(
    len(forecast_df) == 7 * 24,
    f"Test 5 — {7*24} hourly rows returned",
    f"Test 5 — wrong count: {len(forecast_df)}"
)

# ── Test 6 — Forecast accuracy reports LZ price accuracy not spread ─────────
accuracy = forecast_accuracy(forecast_df)
check(
    accuracy["mae"] > 0,
    f"Test 6 — MAE = ${accuracy['mae']:.2f}/MWh (LZ price forecast error)",
    "Test 6 — MAE is 0 — suspicious"
)
check(
    accuracy["direction_accuracy"] > 50,
    f"Test 6 — direction accuracy = {accuracy['direction_accuracy']:.1f}% (>50% = better than random)",
    f"Test 6 — direction accuracy {accuracy['direction_accuracy']:.1f}% is no better than random"
)
print(f"\n  Forecast accuracy: MAE=${accuracy['mae']:.2f} | Dir={accuracy['direction_accuracy']:.1f}%\n")

In [ ]:
# ── Test 7 — optimise_dam_bid uses forecast LZ prices not spreads ──────────
forecast_prices = pd.Series(
    [30.0] + [5.0]*4 + [30.0]*14 + [80.0]*4 + [30.0],
    index=range(24)
)
bid = optimise_dam_bid(
    forecast_prices=forecast_prices,
    spec=spec,
    operating_date=pd.Timestamp("2025-06-15"),
)
check(
    hasattr(bid, "forecast_prices"),
    "Test 7 — bid has forecast_prices (LZ prices) not forecast_spreads",
    "Test 7 — bid attribute wrong"
)
check(
    set(bid.charge_hours).issubset(set(range(1, 5))),
    f"Test 7 — charge in cheap hours: {bid.charge_hours}",
    f"Test 7 — charge hours wrong: {bid.charge_hours}"
)
check(
    set(bid.discharge_hours).issubset(set(range(19, 24))),
    f"Test 7 — discharge in expensive hours: {bid.discharge_hours}",
    f"Test 7 — discharge hours wrong: {bid.discharge_hours}"
)
check(
    min(bid.discharge_hours) > max(bid.charge_hours),
    "Test 7 — causal: discharge after charge",
    "Test 7 — causal constraint violated"
)

# ── Test 8 — settle_dam_bid settles at actual LZ_HOUSTON_DAM prices ─────────
actual_prices = pd.Series(
    [30.0] + [8.0]*4 + [30.0]*14 + [70.0]*4 + [30.0],
    index=pd.date_range("2025-06-15", periods=24, freq="h")
)
settlement = settle_dam_bid(bid, actual_prices, spec)

actual_charge_price = settlement["schedule_df"]["actual_lz_houston"].iloc[1]
check(
    actual_charge_price == 8.0,
    f"Test 8 — settlement uses actual LZ price (${actual_charge_price}) not forecast",
    f"Test 8 — wrong settlement price: ${actual_charge_price}"
)
check(
    "forecast_lz_houston"  in settlement["schedule_df"].columns,
    "Test 8 — forecast_lz_houston column present (LZ prices not spreads)",
    "Test 8 — wrong column name"
)
check(
    "actual_lz_houston" in settlement["schedule_df"].columns,
    "Test 8 — actual_lz_houston column present (LZ prices not spreads)",
    "Test 8 — wrong column name"
)
print(f"\n  Settlement: net=${settlement['net_revenue']:,.2f} | "
      f"forecast=${settlement['forecast_revenue']:,.2f} | "
      f"gap=${settlement['realisation_gap']:,.2f}\n")

# ── Test 9 — realisation gap = actual - forecast revenue ──────────────────
expected_gap = settlement["net_revenue"] - settlement["forecast_revenue"]
check(
    abs(settlement["realisation_gap"] - expected_gap) < 0.01,
    f"Test 9 — realisation gap = actual - forecast: ${settlement['realisation_gap']:,.2f}",
    "Test 9 — gap arithmetic wrong"
)

In [ ]:
# ── Test 10 — run_dam_campaign: all revenue at LZ_HOUSTON_DAM ─────────────
campaign_df = run_dam_campaign(forecast_df, df_coast, spec)
check(
    len(campaign_df) > 0,
    f"Test 10 — campaign ran {len(campaign_df)} days",
    "Test 10 — campaign returned 0 rows"
)
check(
    "realisation_gap" in campaign_df.columns,
    "Test 10 — realisation_gap column present",
    "Test 10 — realisation_gap MISSING"
)

# ── Test 11 — dam_campaign_kpis structure ─────────────────────────────────
kpis = dam_campaign_kpis(campaign_df, spec)
required = [
    "total_net_revenue", "avg_daily_revenue", "best_day",
    "worst_day", "profitable_days_pct", "total_soc_violations",
    "total_realisation_gap", "avg_capture_ratio_pct", "revenue_per_mw_year"
]
for k in required:
    check(k in kpis, f"Test 11 — KPI '{k}' present", f"Test 11 — KPI '{k}' MISSING")

print(f"\n  Campaign KPIs:")
for k, v in kpis.items():
    print(f"    {k}: {v}")

# ── Test 12 — no Streamlit imports ────────────────────────────────────────
for fname in ["../src/forecaster.py", "../src/dam_bidder.py"]:
    with open(fname) as f:
        src = f.read()
    check(
        "import streamlit" not in src,
        f"Test 12 — no Streamlit imports in {fname.split('/')[-1]}",
        f"Test 12 — Streamlit found in {fname.split('/')[-1]}"
    )

# ── Test L1 — feature lag is 24 hours ─────────────────────────────────────
check(
    model_result.get("feature_lag_hours") == 24,
    "Test L1 — feature_lag_hours = 24 documented in model result",
    f"Test L1 — wrong lag: {model_result.get('feature_lag_hours')}"
)

# ── Test L2 — generate_day_ahead_forecast uses D-1 features ───────────────
dataset_start = pd.Timestamp(df_coast.index.min().date())
try:
    bad_fcst = generate_day_ahead_forecast(
        model_result, df_coast,
        operating_date=dataset_start
    )
    print("❌ Test L2 — should have raised ValueError for missing D-1 data")
    failed += 1
except ValueError as e:
    check(True,
          f"Test L2 — correctly raises ValueError when D-1 missing: '{str(e)[:50]}'",
          "")

# ── Test L3 — forecast index is Day D not Day D-1 ─────────────────────────
operating_date = pd.Timestamp("2025-06-15")
fcst = generate_day_ahead_forecast(model_result, df_coast, operating_date)

check(
    fcst.index.min().date() == operating_date.date(),
    f"Test L3 — forecast index is Day D ({operating_date.date()}) not D-1",
    f"Test L3 — forecast index is wrong date: {fcst.index.min().date()}"
)
check(
    len(fcst) == 24,
    f"Test L3 — 24 forecast values for Day D",
    f"Test L3 — wrong count: {len(fcst)}"
)

# ── Test L4 — no D-1 feature values appear in forecast output ─────────────
d_minus_1 = operating_date - pd.Timedelta(days=1)
check(
    d_minus_1.date() not in list(fcst.index.date),
    "Test L4 — D-1 timestamps not in forecast output index",
    "Test L4 — D-1 timestamps found in forecast index — indexing bug"
)

# ── Test L5 — training features are lagged (first 24 rows dropped) ─────────
total_rows = len(df_coast.dropna(subset=[BATTERY_LZ_COL]))
modelled   = model_result["train_size"] + model_result["test_size"]
check(
    modelled <= total_rows - 24,
    f"Test L5 — {modelled:,} modelled rows ≤ {total_rows - 24:,} "
    f"(total - 24h lag drop): lag applied correctly",
    f"Test L5 — {modelled:,} rows suggests lag not applied: "
    f"expected ≤ {total_rows - 24:,}"
)

# ── Test L6 — MAE is higher than before lag (honest model has real error) ─
check(
    model_result["mae"] > 1.0,
    f"Test L6 — MAE = ${model_result['mae']:.2f} > $1.00 "
    "(confirms no trivial lookahead — honest model has real error)",
    f"Test L6 — MAE = ${model_result['mae']:.2f} is suspiciously low — "
    "check for remaining lookahead"
)

print(f"\n{'─'*50}")
print(f"Results: {passed} passed / {failed} failed / {passed+failed} total")
if failed == 0:
    print("🎉 All tests passed — ready for Step 5 (Forecast Error Analysis)")
else:
    print("⚠️  Fix failing tests before proceeding")
print(f"{'─'*50}")